# Vessel card — evidence and preventive action

Notebook 02 produces `V001, 85.5, High, [Repetition, Trend, Unresolved, Maintenance]`. Nobody can
act on that. A superintendent reads it and asks what to actually do, and the score has no answer.

This notebook closes that gap. It takes each vessel's drivers and attaches:

- **the source records behind them** — which deficiency IDs, which audit finding, which overdue job
- **a recommended action, an owning department and a target window**

Both layers are lookups, not models. Nothing here re-derives the score, and the score does not
depend on anything here. That separation is the point: a superintendent who disagrees with a
recommendation can check the underlying records without questioning the ranking, and a disputed
score can be investigated without touching the actions.

The output is the eight-column view the brief asks for in Section 6:

> Current Risk → Why → Evidence → Critical Issues → Action → Ownership → Timing → Readiness

### 0. Setup

In [1]:
import sys
from pathlib import Path

for _dir in (Path.cwd(), *Path.cwd().parents):
    if (_dir / 'pyproject.toml').exists():
        sys.path.insert(0, str(_dir))
        break

import pandas as pd

from feature_builder import build_features, REFERENCE_DATE
from scoring import score_from_features, DIMENSIONS
from vessel_card import ACTIONS, evidence_for, since_last_inspection, recommendations_for

pd.set_option('display.width', 200)

features = build_features()
results = score_from_features(features)

print(f'{len(results)} vessels scored as at {REFERENCE_DATE.date()}')
print(results.tier.value_counts().reindex(['High', 'Medium', 'Low']).to_string())

20 vessels scored as at 2026-08-24
tier
High       2
Medium     2
Low       16


## 1. The evidence layer

The score says a vessel is high risk. This layer says *on what basis* — by going back to the
original records and naming them.

Each driver gets its own lookup, and each returns the actual record IDs, not just a sentence.
That matters: a superintendent can open DEF0001, read the finding, and decide the system has it
wrong. Evidence nobody can check is just an opinion.

| Driver | What it goes and finds |
|---|---|
| Repetition | the defect that keeps coming back, how often, and which deficiency records |
| Trend | how many findings at each inspection, oldest to newest |
| Unresolved | what is still open, what was already open before the last inspection, and what the company had already flagged internally |
| Maintenance | which jobs are overdue and how long the oldest has been sitting |
| Concentration | which category the findings cluster in, and what share |
| Crew | who joined recently and the average experience on board |
| Equipment | which items have failed more than once |

In [2]:
v = 'V001'
row = results.loc[v]

print(f'{v} — score {row.score}, tier {row.tier}, drivers {row.drivers}\n')
for driver in row.drivers:
    ev = evidence_for(v, driver)
    print(f'  {driver}')
    print(f'    {ev["summary"]}')
    print(f'    records: {ev["record_ids"]}\n')

V001 — score 85.5, tier High, drivers ['Repetition', 'Trend', 'Unresolved', 'Maintenance']

  Repetition
    "Fire pump pressure low" recorded 9 times across 4 inspections
    records: ['DEF0001', 'DEF0002', 'DEF0003', 'DEF0005', 'DEF0006']

  Trend
    3 finding(s) at 2024-02-15 rising to 10 at 2025-06-29
    records: ['INS0001', 'INS0002', 'INS0003', 'INS0004']

  Unresolved
    10 deficiencies still open, 3 of them raised before the most recent inspection; 4 internal audit findings unresolved, 1 matching an open deficiency
    records: ['DEF0010', 'DEF0011', 'DEF0012', 'DEF0019', 'DEF0020', 'AUD0001', 'AUD0002', 'AUD0003', 'AUD0004']

  Maintenance
    11 planned jobs overdue, oldest due 2026-04-18 (128 days ago)
    records: ['MAIN0001', 'MAIN0002', 'MAIN0003', 'MAIN0004', 'MAIN0005']



The Unresolved line is the one worth reading twice. Three of V001's ten open deficiencies were
raised **before its most recent inspection** — meaning an inspector had already seen and recorded
them once, they were not closed, and they were still outstanding when the next inspector boarded.
Verifying rectification of previous findings is specifically what a PSC officer does.

That feature was excluded from the score in notebook 02 because it is non-zero for exactly one
vessel and therefore useless in a weighted average. It is the most damning fact available about
this vessel. **Useless for ranking, decisive for explaining** — which is the whole argument for
keeping the two layers separate.

## 2. Critical issues — what has changed since the last inspection

Every vessel in the fleet is **415–455 days** past its last inspection. The inspection-derived
features therefore describe each vessel as it was more than a year ago, while the operational
tables describe it now.

Nothing in this block was visible to the last inspector. All of it will be visible to the next
one. That is the gap an early-warning system exists to cover, and it is built entirely from the
features that showed no measurable correlation with historical deficiency counts.

In [3]:
sli = since_last_inspection(v)
print(f'{v} — last inspected {sli["last_inspection"]}, {sli["days_since"]} days ago\n')
for k in ['equipment_failures', 'deficiencies_still_open', 'overdue_maintenance',
          'unresolved_audit_findings', 'crew_joined']:
    print(f'  {k.replace("_", " "):30} {sli[k]}')
print(f'\n  repeat offenders: {sli["equipment_detail"]}')

V001 — last inspected 2025-06-29, 421 days ago

  equipment failures             4
  deficiencies still open        10
  overdue maintenance            11
  unresolved audit findings      4
  crew joined                    9

  repeat offenders: {'Emergency Generator': 2, 'Fire Pump': 1, 'Oil Water Separator': 1}


In [4]:
# fleet-wide, to show the window is never empty
fleet = pd.DataFrame({vid: since_last_inspection(vid) for vid in results.index}).T
totals = fleet[['equipment_failures', 'deficiencies_still_open',
                'overdue_maintenance', 'unresolved_audit_findings']].sum()
print('accumulated across the fleet since each vessel was last inspected:')
print(totals.to_string())
print(f'\ndays since last inspection: {fleet.days_since.min()} to {fleet.days_since.max()}')

accumulated across the fleet since each vessel was last inspected:
equipment_failures           130
deficiencies_still_open       47
overdue_maintenance          121
unresolved_audit_findings     78

days since last inspection: 415 to 455


## 3. Preventive action

Each driver maps to an action, an owning function and a target window. The owning functions are
those named in the brief — Marine, Technical, Crewing, Operations, Vessel.

Priority follows the driver order, which is points descending, so the first task is the largest
single contributor to the score. Urgency comes from the tier rather than the driver: the same
overdue maintenance backlog is immediate on a High vessel and routine on a Low one.

**These mappings are placeholders for company procedure.** In deployment the action text and the
windows would come from the operator's own SMS, not from this notebook. What matters is the
mechanism — driver in, owned and dated task out — not the specific wording.

In [5]:
print(pd.DataFrame(ACTIONS).T[['owner', 'window']].to_string())

                            owner                   window
Repetition              Technical    Before next port call
Trend                      Marine                  30 days
Unresolved     Vessel / Technical                  14 days
Maintenance             Technical                  30 days
Concentration              Marine                  30 days
Crew                      Crewing  Before next crew change
Equipment               Technical                  30 days


In [6]:
for r in recommendations_for(row.drivers, row.tier):
    print(f'{r["priority"]}. [{r["urgency"]}] {r["driver"]}  ->  {r["owner"]}, {r["window"]}')
    print(f'   {r["action"]}\n')

1. [Immediate] Repetition  ->  Technical, Before next port call
   Root-cause the recurring defect rather than re-closing it. Verify the last rectification actually held, and record objective evidence of the repair.

2. [Immediate] Trend  ->  Marine, 30 days
   Review why inspection outcomes are deteriorating. Superintendent attendance at the next port, with a pre-inspection walkthrough against the last report.

3. [Immediate] Unresolved  ->  Vessel / Technical, 14 days
   Close outstanding findings and produce rectification evidence. Findings open across more than one inspection are the first thing a PSC officer verifies.

4. [Immediate] Maintenance  ->  Technical, 30 days
   Clear the overdue planned maintenance backlog, prioritising statutory and safety-critical equipment. Escalate any job that cannot be closed in window.



## 4. The vessel card

All eight output areas assembled into one view. This is the deliverable the brief asks for.

**Readiness** is the one field that is not a lookup. It answers "what would this vessel look like
if the recommended actions were completed?" — computed by clearing the closable items and
re-scoring. It is shown as a projection, not a promise, and the gap between current and projected
is itself informative: if closing everything actionable barely moves the score, the problem is
structural rather than a backlog.

In [7]:
from scoring import normalise, prepare, score_fleet, rank_fleet, SCORING_INPUTS

CLOSABLE = ['open_deficiencies', 'overdue_maintenance', 'open_audit', 'overdue_audit',
            'known_issues_unresolved', 'known_issue_occurrences']

# min-max bounds from the fleet as it stands today, held fixed for readiness
_prepared = prepare(features)
_BOUNDS = {c: (_prepared[c].min(), _prepared[c].max())
           for c in SCORING_INPUTS if c in _prepared.columns}


def readiness(vessel_id, features=features):
    '''Re-score one vessel with its actionable items cleared, on today's fleet scale.

    Only genuinely closable things are zeroed. History cannot be closed: repetition, trend
    and concentration describe inspections that already happened and stay as they are.

    The bounds are deliberately frozen. Re-normalising after the edit would rescale the
    whole fleet - removing V001's extreme values lifts every other vessel's score, and V004
    would appear to jump from 60.6 to 72.5 without anything changing aboard it. That is a
    true statement about a fleet-relative score and a useless answer to the question the
    card is asking, which is what THIS vessel would look like on today's scale.
    '''
    f = features.copy()
    f.loc[vessel_id, CLOSABLE] = 0
    p = prepare(f)

    norm = pd.DataFrame({
        c: ((p[c] - lo) / (hi - lo)).clip(0, 1) if hi > lo else pd.Series(0.0, index=p.index)
        for c, (lo, hi) in _BOUNDS.items()})

    scored = score_fleet(norm)
    return rank_fleet(scored).loc[vessel_id, ['score', 'tier']]


def vessel_card(vessel_id, results=results):
    r = results.loc[vessel_id]
    p = features.loc[vessel_id]
    proj = readiness(vessel_id)
    sli = since_last_inspection(vessel_id)

    print('=' * 78)
    print(f'{vessel_id}  {p.vessel_name}   {p.vessel_type}, {p.flag_state}, built {p.build_year}')
    print('=' * 78)

    print(f'\nCURRENT RISK   {r.score} / 100   tier {r.tier}   rank {r["rank"]} of {len(results)}')

    if not r.drivers:
        print('\nWHY            No dimension above the fleet 75th percentile. This vessel is not')
        print('               an outlier on any measure - see Critical Issues for current state.')
    else:
        print('\nWHY            ' + ', '.join(r.drivers))
        if r.saturated:
            print('               at maximum on: ' + ', '.join(r.saturated))

    print('\nEVIDENCE')
    # with no elevated driver, fall back to this vessel's own two largest dimensions so the
    # evidence still describes the vessel rather than an arbitrary pair
    fallback = r[list(DIMENSIONS)].astype(float).nlargest(2).index.tolist()
    for driver in (r.drivers or fallback):
        ev = evidence_for(vessel_id, driver)
        if ev:
            print(f'   {driver:14} {ev["summary"]}')
            print(f'   {"":14} records: {ev["record_ids"][:4]}')

    print(f'\nCRITICAL ISSUES   since last inspection {sli["last_inspection"]} '
          f'({sli["days_since"]} days)')
    print(f'   {sli["equipment_failures"]} equipment failures, '
          f'{sli["deficiencies_still_open"]} deficiencies still open, '
          f'{sli["overdue_maintenance"]} overdue jobs, '
          f'{sli["unresolved_audit_findings"]} unresolved audit findings')

    recs = recommendations_for(r.drivers, r.tier)
    if recs:
        print('\nACTION / OWNERSHIP / TIMING')
        for x in recs:
            print(f'   {x["priority"]}. {x["driver"]:14} {x["owner"]:20} {x["window"]:26} '
                  f'[{x["urgency"]}]')
            print(f'      {x["action"]}')
    else:
        print('\nACTION / OWNERSHIP / TIMING')
        print('   No driver-led action. Clear the overdue maintenance backlog and close')
        print('   outstanding audit findings as routine work.  Technical / Vessel, 30 days.')

    print(f'\nREADINESS      {r.score} now  ->  {proj.score} if the closable items above are '
          f'cleared  (tier {proj.tier})')
    print('=' * 78)


vessel_card('V001')

V001  MV Alpha   Bulk Carrier, Panama, built 2008

CURRENT RISK   85.5 / 100   tier High   rank 1 of 20

WHY            Repetition, Trend, Unresolved, Maintenance
               at maximum on: Repetition, Trend, Maintenance

EVIDENCE
   Repetition     "Fire pump pressure low" recorded 9 times across 4 inspections
                  records: ['DEF0001', 'DEF0002', 'DEF0003', 'DEF0005']
   Trend          3 finding(s) at 2024-02-15 rising to 10 at 2025-06-29
                  records: ['INS0001', 'INS0002', 'INS0003', 'INS0004']
   Unresolved     10 deficiencies still open, 3 of them raised before the most recent inspection; 4 internal audit findings unresolved, 1 matching an open deficiency
                  records: ['DEF0010', 'DEF0011', 'DEF0012', 'DEF0019']
   Maintenance    11 planned jobs overdue, oldest due 2026-04-18 (128 days ago)
                  records: ['MAIN0001', 'MAIN0002', 'MAIN0003', 'MAIN0004']

CRITICAL ISSUES   since last inspection 2025-06-29 (421 days)
   4 equipme

### Reading the V001 card

Score and tier say *look here first*. The drivers say *why*. The evidence says *which records*,
so a superintendent can open DEF0010 and check for themselves. The actions say *who does what by
when*.

**Readiness is the interesting number.** Closing every closable item moves V001 substantially but
does not clear it, because Repetition, Trend and Concentration describe inspections that already
happened and cannot be closed. That is the correct answer, and it is a useful thing to tell a
manager: this vessel cannot be fixed by clearing a backlog, because its risk is a history of
recurring defects rather than a queue of open jobs.

## 5. A vessel with no drivers

Nine of twenty vessels have no dimension above the fleet's 75th percentile. Sixteen are Low tier.
A system that only produces a useful card for the top two is not an operational tool — the brief
asks that a user opening **one vessel** understands its position, not that a user opening one of
the two worst vessels does.

V016 is the case that makes the point. It scores 25.1, exactly the same as V013, and returns no
drivers at all where V013 returns two.

In [8]:
vessel_card('V016')

V016  MV Voyager   Bulk Carrier, Panama, built 2023

CURRENT RISK   25.1 / 100   tier Low   rank 5 of 20

WHY            No dimension above the fleet 75th percentile. This vessel is not
               an outlier on any measure - see Critical Issues for current state.

EVIDENCE
   Maintenance    7 planned jobs overdue, oldest due 2026-02-08 (197 days ago)
                  records: ['MAIN0306', 'MAIN0312', 'MAIN0313', 'MAIN0314']
   Trend          1 finding(s) at 2024-02-02 rising to 3 at 2025-06-13
                  records: ['INS0061', 'INS0062', 'INS0063', 'INS0064']

CRITICAL ISSUES   since last inspection 2025-06-13 (437 days)
   6 equipment failures, 1 deficiencies still open, 7 overdue jobs, 4 unresolved audit findings

ACTION / OWNERSHIP / TIMING
   No driver-led action. Clear the overdue maintenance backlog and close
   outstanding audit findings as routine work.  Technical / Vessel, 30 days.

READINESS      25.1 now  ->  14.0 if the closable items above are cleared  (tier Low)

**Same score, different card, and correctly so.** V013 has two specific elevated dimensions and a
driver-led action list. V016 has nothing unusual in its inspection record — but it is not empty,
because the Critical Issues block still reports what has accumulated since it was last inspected.

The wording matters here. A Low-tier card says *nothing unusual in the inspection record*, never
*this vessel is safe*. The distinction is not pedantry: for a newly acquired vessel with no
inspection history the score would also be low, and would mean nothing is known rather than
nothing is wrong. Presenting a low score as a clearance is the failure mode this system most needs
to avoid.

## 6. Fleet view

The same information for every vessel, as an operations user would first see it.

In [9]:
fleet_view = pd.DataFrame({
    'vessel': features.vessel_name,
    'score': results.score,
    'tier': results.tier,
    'why': results.drivers.apply(lambda d: ', '.join(d) if d else 'no elevated dimension'),
    'first_action': results.drivers.apply(
        lambda d: ACTIONS[d[0]]['owner'] + ' / ' + ACTIONS[d[0]]['window'] if d else 'routine'),
}).sort_values('score', ascending=False)

print(fleet_view.to_string())

               vessel  score    tier                                           why                       first_action
vessel_id                                                                                                            
V001         MV Alpha   85.5    High    Repetition, Trend, Unresolved, Maintenance  Technical / Before next port call
V004         MV Delta   60.6    High  Concentration, Trend, Unresolved, Repetition                   Marine / 30 days
V019         MV Atlas   32.2  Medium      Maintenance, Equipment, Repetition, Crew                Technical / 30 days
V007       MV Horizon   29.4  Medium            Maintenance, Repetition, Equipment                Technical / 30 days
V013         MV Titan   25.1     Low                        Unresolved, Repetition       Vessel / Technical / 14 days
V016       MV Voyager   25.1     Low                         no elevated dimension                            routine
V014         MV Unity   25.0     Low                    

In [10]:
# the full card set, written out so the whole fleet is reproducible from this notebook
from pathlib import Path
import io, contextlib

out_path = Path.cwd()
for _dir in (Path.cwd(), *Path.cwd().parents):
    if (_dir / 'pyproject.toml').exists():
        out_path = _dir
        break

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    for vid in results.index:
        vessel_card(vid)
        print()

(out_path / 'vessel_cards.txt').write_text(buf.getvalue())
fleet_view.to_csv(out_path / 'fleet_view.csv')
print(f'vessel_cards.txt   {len(buf.getvalue().splitlines())} lines, all {len(results)} vessels')
print(f'fleet_view.csv     {fleet_view.shape[0]} rows')

vessel_cards.txt   543 lines, all 20 vessels
fleet_view.csv     20 rows


## 7. Where generative AI fits, and where it does not

Everything above is deterministic — arithmetic, threshold comparisons and table lookups. The same
inputs produce the same card every time, and every line traces to a record ID.

A generative model has exactly one job in this design: **turning the finished card into readable
prose.** It receives the structured object, which is already complete, and writes it as English for
a management summary or an email to a superintendent. It selects nothing, ranks nothing and decides
nothing.

That placement is deliberate rather than cautious. If the model chose the drivers or wrote the
actions, the traceability the evidence layer exists to provide would be gone, and the brief's
requirement to keep *risk prediction, supporting evidence, and recommended actions clearly
separated* would be broken.

**Which is the answer to the question the brief asks directly.** If the generative component is
unavailable, the system still scores, ranks, tiers, identifies drivers, retrieves evidence,
assigns actions, owners and target windows, and computes readiness. It loses polish, not function.

The prompt below is shown but deliberately not called, so this notebook reproduces without an API
key or network access.

In [11]:
def narrative_prompt(vessel_id):
    '''Build the grounded prompt. Every fact the model is allowed to use is passed in.'''
    r = results.loc[vessel_id]
    payload = {
        'vessel': features.loc[vessel_id, 'vessel_name'],
        'score': float(r.score), 'tier': str(r.tier), 'rank': int(r['rank']),
        'drivers': list(r.drivers),
        'evidence': {d: evidence_for(vessel_id, d)['summary']
                     for d in r.drivers if evidence_for(vessel_id, d)},
        'since_last_inspection': since_last_inspection(vessel_id),
        'actions': recommendations_for(r.drivers, r.tier),
    }
    instruction = (
        'Write three sentences for a fleet manager, using ONLY the facts in the JSON below. '
        'Do not introduce any number that does not appear there. Do not infer causes. '
        'Do not soften or dramatise. If a fact is absent, omit it rather than guessing.'
    )
    return instruction, payload


instruction, payload = narrative_prompt('V001')
print(instruction, '\n')
print({k: payload[k] for k in ['vessel', 'score', 'tier', 'drivers']})
print('\n(not called - the notebook runs without credentials. Every number in any generated')
print(' output must be verifiable against this payload; that check is the acceptance test.)')

Write three sentences for a fleet manager, using ONLY the facts in the JSON below. Do not introduce any number that does not appear there. Do not infer causes. Do not soften or dramatise. If a fact is absent, omit it rather than guessing. 

{'vessel': 'MV Alpha', 'score': 85.5, 'tier': 'High', 'drivers': ['Repetition', 'Trend', 'Unresolved', 'Maintenance']}

(not called - the notebook runs without credentials. Every number in any generated
 output must be verifiable against this payload; that check is the acceptance test.)


## Summary

- The score answers *which vessel*. This notebook answers *why, on what evidence, and what to do
  first* — the remaining three parts of the business question.
- **Every line on a card carries its record IDs.** Evidence that cannot be opened and checked is
  an assertion, and the brief asks specifically for traceability to source.
- Actions carry an owning function and a target window. Priority follows driver size; urgency
  follows tier.
- **Readiness separates what can be fixed from what cannot.** Closing V001's backlog moves the
  score but does not clear it, because a history of recurring defects is not a queue of open jobs.
- Cards are produced for all twenty vessels, including the nine with no elevated dimension, where
  the card reports what has accumulated since the last inspection instead. A low score is never
  presented as a clearance.
- Generative AI sits at the end, converting a finished card into prose. Remove it and the system
  loses readability, not capability.